# Group30: all 15 MVTec AD categories

Fixed 224px comparison with verified stage-level resume. Reuses the completed bottle experiment. Run all cells in order after reconnecting; training within an interrupted stage restarts.


In [ ]:
import datetime
import json
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/PRS_Group30')
SOURCE_ZIP = BASE / 'Visual_Defect_Detection_GPU.zip'
DATA_ARCHIVE = BASE / 'mvtec_anomaly_detection.tar.xz'
CATEGORIES = ['bottle']  # All categories: bottle, cable, capsule, carpet, grid, hazelnut, leather, metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper
EPOCHS = 10
IMAGE_SIZE = 128
BATCH_SIZE = 16
SEED = 42
assert SOURCE_ZIP.is_file(), f'Upload source ZIP: {SOURCE_ZIP}'
assert DATA_ARCHIVE.is_file(), f'Upload dataset: {DATA_ARCHIVE}'


## Install the project and check GPU availability

In [ ]:
PROJECT = Path('/content/Visual_Defect_Detection_GPU')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    destination = PROJECT.parent if all(name.startswith(PROJECT.name + "/") for name in archive.namelist()) else PROJECT
    for name in archive.namelist():
        if not (destination / name).resolve().is_relative_to(destination.resolve()):
            raise ValueError(f'Unsafe ZIP path: {name}')
    archive.extractall(destination)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[dev]'], check=True)
import torch

print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime and run this cell again.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, check=True)


## Extract the dataset
Initial extraction may take a while. Training reads images from the Colab local disk rather than Google Drive.

In [ ]:
LOCAL_ARCHIVE = Path('/content/mvtec_anomaly_detection.tar.xz')
DATA_DIR = Path('/content/mvtec_data')
MARKER = DATA_DIR / '.extraction_complete'
if not MARKER.exists():
    DATA_DIR.mkdir(exist_ok=True)
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)
    with tarfile.open(LOCAL_ARCHIVE) as archive:
        archive.extractall(DATA_DIR, filter='data')
    MARKER.touch()
roots = [DATA_DIR] + [p for p in DATA_DIR.iterdir() if p.is_dir()]
matches = [p for p in roots if (p / 'bottle/train/good').is_dir()]
assert len(matches) == 1, 'Cannot identify MVTec AD dataset root'
DATA_ROOT = matches[0]
for category in CATEGORIES:
    assert (DATA_ROOT / category / 'train/good').is_dir(), category
    assert (DATA_ROOT / category / 'test').is_dir(), category
print('Dataset:', DATA_ROOT)


In [ ]:
# Re-run this cell unchanged to resume completed stages after interruption.
REMOTE = BASE / 'results/full15_224_v1'
REMOTE.mkdir(parents=True, exist_ok=True)
if not (REMOTE / 'source.zip').exists():
    shutil.copy2(SOURCE_ZIP, REMOTE / 'source.zip')
command = [sys.executable, '-u', '-m', 'vdd.suite',
    '--root', str(DATA_ROOT), '--remote', str(REMOTE),
    '--local', '/content/vdd_runs/full15_224_v1',
    '--reuse-bottle', str(BASE / 'results/benchmark224_20261030_140519')]
subprocess.run([sys.executable, '-m', 'pip', 'install', 'anomalib==2.7.0'], check=True)
process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='', flush=True)
assert process.wait() == 0, 'Some stages failed: review status.json and rerun unchanged to resume.'


In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'notebooks/analyze_suite.py'), '--run', str(REMOTE), '--root', str(DATA_ROOT)], check=True)
